#Downloading the PDF file:

In [4]:
from google.colab import files

uploaded = files.upload()

Saving sample_ner_with_table.pdf to sample_ner_with_table.pdf


#Using PyMuPDF to extract text from the PDF file:

In [6]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 57.3 MB/s eta 0:00:00


In [15]:
import fitz  # PyMuPDF

# Replace with your uploaded file name
pdf_file = list(uploaded.keys())[0]

# Open the PDF
doc = fitz.open(pdf_file)

# Extract text
text = ""

for page in doc:
    text += page.get_text().replace("\n", " ")

doc.close()

# Print result
print('text =',text)

text = John Smith works at OpenAI in San Francisco. He met Maria Garcia from Shopify in Toronto on March 10, 2023. They discussed collaborations with Amazon Web Services and Google Cloud in New York City. Ali Rezaei from Digikala in Tehran also joined the virtual meeting. Name Organization Location Date John Smith OpenAI San Francisco, USA 2023-03-10 Maria Garcia Shopify Toronto, Canada 2022-06-15 Ali Rezaei Digikala Tehran, Iran 2021-11-05 Emma Brown Google New York City, USA 2023-01-20 


#Running custome NER with the defult model ("dslim/bert-base-NER")

In [38]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
import pandas as pd
import torch.nn.functional as F


class CustomNER:
    def __init__(self, model_name="dslim/bert-base-NER"):
        """
        Initialize with a specific NER model
        """
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

        # Get label mappings
        self.id2label = self.model.config.id2label

    def predict(self, text):
        """
        Run NER prediction on text
        """
        # Tokenize input
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Get predictions
        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            probs = F.softmax(logits, dim=-1)

        # Get predictions
        predictions = torch.argmax(outputs.logits, dim=2)
        scores = torch.max(probs, dim=-1).values

        # Convert tokens and predictions
        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        predictions = predictions[0].cpu().numpy()

        # Create results
        results = []
        current_entity = None

        for token, pred_id, score in zip(tokens, predictions, scores[0]):
            label = self.id2label[pred_id]

            # Skip special tokens
            if token in ["[CLS]", "[SEP]", "[PAD]"]:
                continue

            # Handle subwords
            if token.startswith("##"):
                token = token[2:]
                if current_entity:
                    current_entity["word"] += token
                    continue

            # If not continuing previous entity
            if current_entity:
                results.append(current_entity)

            if label != "O":  # O means no entity
                current_entity = {
                    "word": token,
                    "entity": label,
                    "score": float(score.item())
                }
            else:
                current_entity = None

        # Add last entity
        if current_entity:
            results.append(current_entity)

        return results

# Usage example
def run_custom_ner():
    ner = CustomNER()

    results = ner.predict(text)

    # Display as DataFrame
    df = pd.DataFrame(results)
    print(df)

run_custom_ner()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


         word entity     score
0        John  B-PER  0.999600
1       Smith  I-PER  0.999547
2      OpenAI  B-ORG  0.998825
3         San  B-LOC  0.999594
4   Francisco  I-LOC  0.999471
5       Maria  B-PER  0.999591
6      Garcia  I-PER  0.999636
7     Shopify  B-ORG  0.998858
8     Toronto  B-LOC  0.999585
9      Amazon  B-ORG  0.999372
10        Web  I-ORG  0.999219
11   Services  I-ORG  0.999021
12     Google  B-ORG  0.999140
13      Cloud  I-ORG  0.998457
14        New  B-LOC  0.999539
15       York  I-LOC  0.999469
16       City  I-LOC  0.999617
17        Ali  B-PER  0.999707
18     Rezaei  I-PER  0.999404
19   Digikala  B-ORG  0.998705
20     Tehran  B-LOC  0.999734
21       John  B-PER  0.535536
22      Smith  I-ORG  0.475192
23         AI  I-ORG  0.913450
24        San  B-LOC  0.941811
25  Francisco  I-LOC  0.999350
26        USA  B-LOC  0.999713
27      Maria  B-PER  0.993017
28     Garcia  I-PER  0.988554
29    Shopify  I-PER  0.764775
30    Toronto  I-LOC  0.769538
31     C

#Running custome NER with "xlm-roberta-large-finetuned-conll03-english" model

In [39]:
def run_custom_ner():
    ner = CustomNER(model_name="xlm-roberta-large-finetuned-conll03-english")

    results = ner.predict(text)

    # Display as DataFrame
    df = pd.DataFrame(results)
    print(df)

run_custom_ner()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-large-finetuned-conll03-english
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


          word entity     score
0        ▁John  I-PER  0.999997
1       ▁Smith  I-PER  0.999997
2        ▁Open  I-ORG  0.999995
3           AI  I-ORG  0.999993
4         ▁San  I-LOC  0.999993
5   ▁Francisco  I-LOC  0.999991
6       ▁Maria  I-PER  0.999987
7      ▁Garcia  I-PER  0.999983
8        ▁Shop  I-ORG  0.999978
9          ify  I-ORG  0.999986
10    ▁Toronto  I-LOC  0.999988
11     ▁Amazon  I-ORG  0.999985
12        ▁Web  I-ORG  0.999974
13   ▁Services  I-ORG  0.999985
14     ▁Google  I-ORG  0.999921
15      ▁Cloud  I-ORG  0.999513
16        ▁New  I-LOC  0.999997
17       ▁York  I-LOC  0.999996
18       ▁City  I-LOC  0.999995
19        ▁Ali  I-PER  0.999997
20         ▁Re  I-PER  0.999997
21          za  I-PER  0.999992
22          ei  I-PER  0.999991
23       ▁Digi  I-ORG  0.999971
24        kala  I-ORG  0.999969
25     ▁Tehran  I-LOC  0.999994
26       ▁John  I-PER  0.999995
27      ▁Smith  I-PER  0.999993
28       ▁Open  I-ORG  0.999994
29          AI  I-ORG  0.999990
30      

#Webpage to NER

In [72]:
pip install readability-lxml

In [73]:
# Extract clean text from a webpage

from readability import Document

def extract_readable_text(url):
    response = requests.get(url)
    doc = Document(response.text)

    soup = BeautifulSoup(doc.summary(), "lxml")
    return soup.get_text(separator=" ")

In [74]:
ner_model = CustomNER()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [129]:
url = "https://www.theguardian.com/world/2026/mar/18/ancient-skeleton-discovered-sitting-upright-in-france?utm_source=firefox-newtab-en-ca"

text = extract_main_text(url)

results = ner_model.predict(text)

results

[{'word': 'Gauls', 'entity': 'B-MISC', 'score': 0.9966289401054382},
 {'word': 'France', 'entity': 'B-LOC', 'score': 0.9996371269226074},
 {'word': 'Dijon', 'entity': 'B-LOC', 'score': 0.9990481734275818},
 {'word': 'Gauls', 'entity': 'B-MISC', 'score': 0.9975180625915527},
 {'word': 'Josephine', 'entity': 'B-PER', 'score': 0.9685940742492676},
 {'word': 'Baker', 'entity': 'I-PER', 'score': 0.7741456031799316},
 {'word': 'Dijon', 'entity': 'B-LOC', 'score': 0.9974130988121033},
 {'word': 'Gauls', 'entity': 'B-MISC', 'score': 0.9609257578849792},
 {'word': 'Celtic', 'entity': 'B-MISC', 'score': 0.9994496703147888},
 {'word': 'French', 'entity': 'B-MISC', 'score': 0.9992747902870178},
 {'word': 'Asterix', 'entity': 'B-ORG', 'score': 0.847586989402771},
 {'word': 'Obelix', 'entity': 'B-LOC', 'score': 0.21420396864414215},
 {'word': 'Gauls', 'entity': 'B-MISC', 'score': 0.955168604850769},
 {'word': 'BC', 'entity': 'B-MISC', 'score': 0.9624782204627991},
 {'word': 'France', 'entity': 'B-LO

##Save to CSV

In [121]:
# Save to CSV

import requests
from bs4 import BeautifulSoup
import csv

def extract_main_text(url):
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(response.content, "lxml")

    for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside"]):
        tag.extract()

    paragraphs = soup.find_all("p")
    text = " ".join(p.get_text() for p in paragraphs)
    return " ".join(text.split())

def split_text(text, max_length=400):
    words = text.split()
    return [" ".join(words[i:i+max_length]) for i in range(0, len(words), max_length)]

def group_entities(results):
    grouped = []
    current_entity, current_label, scores = [], None, []

    for r in results:
        label = r["entity"]
        if label.startswith("B-"):
            if current_entity:
                grouped.append({"word": " ".join(current_entity),
                                "entity": current_label,
                                "score": sum(scores)/len(scores)})
            current_entity = [r["word"]]
            current_label = label[2:]
            scores = [r["score"]]
        elif label.startswith("I-") and current_entity:
            current_entity.append(r["word"])
            scores.append(r["score"])
        else:
            if current_entity:
                grouped.append({"word": " ".join(current_entity),
                                "entity": current_label,
                                "score": sum(scores)/len(scores)})
            current_entity, current_label, scores = [], None, []

    if current_entity:
        grouped.append({"word": " ".join(current_entity),
                        "entity": current_label,
                        "score": sum(scores)/len(scores)})
    return grouped

def save_grouped_to_csv(results, filename="ner_results.csv"):
    grouped = group_entities(results)
    with open(filename, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["word", "entity", "score"])
        writer.writeheader()
        writer.writerows(grouped)
    print(f"Saved {len(grouped)} entities to {filename}")

# --- Main pipeline ---
def process_url_to_csv(url, ner_model, filename="ner_results.csv"):
    text = extract_main_text(url)
    chunks = split_text(text)
    all_results = []
    for chunk in chunks:
        all_results.extend(ner_model.predict(chunk))  # adjust if your method is named differently
    save_grouped_to_csv(all_results, filename)


In [128]:
url = "https://www.theguardian.com/world/2026/mar/18/ancient-skeleton-discovered-sitting-upright-in-france?utm_source=firefox-newtab-en-ca"

process_url_to_csv(url, ner_model)

Saved 32 entities to ner_results.csv


In [127]:
from google.colab import files

files.download('ner_results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>